# Notebook 12: Regime-Conditional Cointegration Analysis

## Objective

This notebook tests whether **cross-country yield spread pairs are cointegrated within specific market regimes**, and if so, whether the resulting spread mean-reverts fast enough to be practically tradable.

The four pairs under analysis are:
- USD–EUR 5Y and 10Y
- JPY–AUD 5Y and 10Y

Regimes (0–3) were identified via a Hidden Markov Model in Notebook 10. Each regime represents a structurally distinct macroeconomic environment (e.g. risk-on, risk-off, high volatility, policy divergence).

## Why Regime-Conditional Testing?

A cointegrating relationship that holds unconditionally across all market environments is of limited trading value — it tells you that the spread *eventually* reverts, but says nothing about *when*. An unconditional test averages over bull markets, crises, and rate cycles simultaneously. If a pair only cointegrates during, say, a low-volatility regime, a trade entered in a high-volatility regime will not revert on any useful timescale.

Regime-conditioning allows us to ask: **"Does this pair behave like a mean-reverting spread *specifically* when the market is in state R?"** If yes, and if regime R persists long enough to realise the trade, then the trade has a structural basis.

## Methodology: Two-Stage Engle-Granger with Spell-Level ADF

### Stage 1A — OLS: Estimating the Regime-Specific Cointegrating Vector

For each regime $r$, we pool all observations labelled as regime $r$ and run:

$$y_{A,t} = \alpha_r + \beta_r \cdot y_{B,t} + \varepsilon_t$$

This gives us the **regime-specific hedge ratio** $\hat{\beta}_r$ and **equilibrium intercept** $\hat{\alpha}_r$.

**Why is pooling non-contiguous observations valid here?**  
OLS is a purely cross-sectional estimator — it minimises the sum of squared residuals across observations regardless of their temporal ordering. The observations do not need to be contiguous. What matters is that the observations all come from the same data-generating process (i.e. the same regime), which is precisely what regime-labelling provides.

The resulting residual $\hat{\varepsilon}_t = y_{A,t} - \hat{\alpha}_r - \hat{\beta}_r \cdot y_{B,t}$ is the **spread** — the component of $y_A$ that is unexplained by the linear relationship with $y_B$ under regime $r$.

---

### Stage 1B — Spell-Level ADF + Fisher Combination: Testing Spread Stationarity

The second step of Engle-Granger tests whether $\hat{\varepsilon}_t$ has a unit root. The **Augmented Dickey-Fuller (ADF)** test asks:

$$\Delta \hat{\varepsilon}_t = \rho \cdot \hat{\varepsilon}_{t-1} + \sum_{j=1}^{k} \gamma_j \Delta \hat{\varepsilon}_{t-j} + u_t$$

- **H₀:** $\rho = 0$ (unit root — the spread wanders without bound, no cointegration)
- **H₁:** $\rho < 0$ (stationary — the spread mean-reverts, cointegration holds)

*Note: The ADF reparametrises the AR(1) level equation $\hat{\varepsilon}_t = \phi \cdot \hat{\varepsilon}_{t-1} + u_t$
by subtracting $\hat{\varepsilon}_{t-1}$ from both sides, yielding $\rho = \phi - 1$.
A unit root ($\phi = 1$) therefore appears as $\rho = 0$ in the ADF equation.

**Why not run ADF directly on all pooled regime observations?**  
The ADF constructs the lagged difference $\Delta \hat{\varepsilon}_t = \hat{\varepsilon}_t - \hat{\varepsilon}_{t-1}$. When observations are non-contiguous (e.g. $t-1$ is the last day of a 2018 spell and $t$ is the first day of a 2022 spell), this difference captures a cross-period jump that is economically meaningless. It corrupts the test statistic and invalidates the ADF critical values.

**The solution: per-spell ADF.** We identify each *contiguous block* (spell) of regime $r$, run ADF within each spell independently (using the same $\hat{\alpha}_r$, $\hat{\beta}_r$ from Stage 1A — no re-estimation), and then aggregate the evidence across spells using **Fisher's combined probability test**:

$$\chi^2_{\text{Fisher}} = -2 \sum_{i=1}^{k} \ln(p_i) \sim \chi^2(2k)$$

where $k$ is the number of spells that met the minimum length threshold. Fisher's method tests the joint null that *all* spells have a unit root. Rejection of this joint null is evidence that cointegration holds recurrently within that regime type — not just in one historical episode.

**Minimum spell length:** 40 observations (~2 months of daily data). Shorter spells are excluded from the ADF step because the ADF has insufficient power at such small samples. They are still included in the OLS step (Stage 1A).

In [25]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / "DATA" / "processed" / "master_df.parquet").exists():
            return cur
        if (cur / ".git").exists():
            return cur
        cur = cur.parent
    raise RuntimeError("Could not find repo root")

project_root = find_repo_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.spreads.cointegration import regime_conditional_cointegration, engle_granger_test
from src.regimes.utils import get_regime_durations
from src.spreads.mean_reversion import estimate_pooled_mean_reversion

data_dir = project_root / 'DATA' / 'processed'
results_dir = project_root / 'results'


In [26]:
# Load master df
df_master = pd.read_parquet(data_dir / 'master_df.parquet')
df_master.index = pd.to_datetime(df_master.index)

# Load regime labels
# Because these are walk-forward, the cointegration tests below are evaluating
# structural validity based solely on what the model "knew" at that point in time.
df_regimes = pd.read_csv(results_dir / 'regime_probabilities.csv', index_col=0, parse_dates=True)

# Merge regime labels into master
df = df_master.join(df_regimes[['regime_most_likely']], how='inner')

print(f"Data range: {df.index.min()} to {df.index.max()}")
print(f"Total observations: {len(df):,}")
print(f"\nRegime distribution:")
print(df['regime_most_likely'].value_counts().sort_index())

Data range: 2006-01-02 00:00:00 to 2025-12-31 00:00:00
Total observations: 4,610

Regime distribution:
regime_most_likely
0    2243
1    1439
2     928
Name: count, dtype: int64


In [27]:
# Adjust accordingly based on pairs tested
pairs = [
    ('USD', 'EUR', '5Y'),
    ('USD', 'EUR', '10Y'),
    ('JPY', 'AUD', '5Y'),
    ('JPY', 'AUD', '10Y')
]

def get_yield_column(ccy: str, tenor: str) -> str:
    """Get bond yield column name from master dataframe."""
    return f'bond_yields__GT{ccy}{tenor} Govt'

# Extract yield series for all pairs
yield_data = {}
for ccy_A, ccy_B, tenor in pairs:
    pair_key = f"{ccy_A}-{ccy_B} {tenor}"
    
    col_A = get_yield_column(ccy_A, tenor)
    col_B = get_yield_column(ccy_B, tenor)
    
    yield_data[pair_key] = {
        'y_A': df[col_A],
        'y_B': df[col_B],
        'ccy_A': ccy_A,
        'ccy_B': ccy_B,
        'tenor': tenor,
        'col_A': col_A,
        'col_B': col_B
    }

### Full 20 year Cointegration per Spread pair

In [28]:
all_test_df = {}
for pair_key, data in yield_data.items():
    print(f"\n{'═'*70}")
    print(f"  UNCONDITIONAL BENCHMARK PAIR: {pair_key}")
    print(f"{'═'*70}")

    test_df = engle_granger_test(
        y_A=data['y_A'],
        y_B=data['y_B'],
        adf_method='aic',
        alpha=0.05
    )
    all_test_df[pair_key] = test_df

    print(f"Cointegrated globally? {test_df['cointegrated']}")
    print(f"ADF p-value: {test_df['adf_pval']:.4f}")


══════════════════════════════════════════════════════════════════════
  UNCONDITIONAL BENCHMARK PAIR: USD-EUR 5Y
══════════════════════════════════════════════════════════════════════
Cointegrated globally? False
ADF p-value: 0.4137

══════════════════════════════════════════════════════════════════════
  UNCONDITIONAL BENCHMARK PAIR: USD-EUR 10Y
══════════════════════════════════════════════════════════════════════
Cointegrated globally? False
ADF p-value: 0.2668

══════════════════════════════════════════════════════════════════════
  UNCONDITIONAL BENCHMARK PAIR: JPY-AUD 5Y
══════════════════════════════════════════════════════════════════════
Cointegrated globally? False
ADF p-value: 0.7236

══════════════════════════════════════════════════════════════════════
  UNCONDITIONAL BENCHMARK PAIR: JPY-AUD 10Y
══════════════════════════════════════════════════════════════════════
Cointegrated globally? False
ADF p-value: 0.5075


## Stage 1 Results: Regime-Conditional Cointegration

The table below is produced for each pair. Column definitions:

| Column | Meaning |
|---|---|
| `n_obs` | Total regime observations used in the OLS step |
| `n_spells_used` | Number of contiguous spells ≥ 40 obs used in the ADF step |
| `hedge_ratio_beta` | $\hat{\beta}_r$ — how many units of $y_B$ hedge one unit of $y_A$ in this regime |
| `alpha` | $\hat{\alpha}_r$ — the equilibrium intercept (spread level at which $\varepsilon = 0$) |
| `r_squared` | OLS fit quality — higher values mean $y_B$ explains more of $y_A$'s level |
| `fisher_chi2` | The combined Fisher test statistic: larger = more evidence against unit root |
| `fisher_pval` | Combined p-value. **Reject H₀ (unit root) if p < 0.05** → cointegrated |
| `cointegrated` | `True` if `fisher_pval < 0.05` |

**Interpreting `hedge_ratio_beta`:** A $\hat{\beta}_r = 0.7$ means that a 1bp move in $y_B$ corresponds to a 0.7bp move in $y_A$ at equilibrium. The spread position would be: long 1 unit of $y_A$, short 0.7 units of $y_B$ (in DV01 terms).

**Interpreting `alpha`:** This is the long-run mean of the spread in this regime. It is regime-specific — the equilibrium level shifts across regimes because the macroeconomic relationship between the two countries changes.

In [43]:
durations_map = get_regime_durations(df['regime_most_likely'])
all_coint_results = {}

for pair_key, data in yield_data.items():
    print(f"\n{'═'*70}")
    print(f"  PAIR: {pair_key}")
    print(f"{'═'*70}")

    coint_df = regime_conditional_cointegration(
        y_A=data['y_A'],
        y_B=data['y_B'],
        regime_labels=df['regime_most_likely'],
        min_obs=100,
        min_spell_obs=40,
        adf_method='aic',
        alpha=0.05
    )
    all_coint_results[pair_key] = coint_df

    # --- Per-regime detailed output ---
    for _, row in coint_df.iterrows():
        r = int(row['regime_state'])
        print(f"\n  ── Regime {r} ──────────────────────────────────────────────")

        # OLS summary
        print(f"  OLS (Stage 1A)")
        print(f"    Observations used : {int(row['n_obs']):,}{'  ⚠ sample warning' if row.get('sample_warning') else ''}")
        if pd.notna(row.get('hedge_ratio_beta')):
            print(f"    Hedge ratio β     : {row['hedge_ratio_beta']:.4f}  (short {row['hedge_ratio_beta']:.4f} units of y_B per unit of y_A)")
            print(f"    Intercept α       : {row['alpha']:.4f}  (regime equilibrium spread level)")
            print(f"    R²                : {row.get('r_squared', float('nan')):.4f}  (proportion of y_A variation explained by y_B)")

        # Fisher ADF summary
        if pd.isna(row.get('fisher_pval')):
            print(f"  Spell ADF (Stage 1B)")
            print(f"    Result  : SKIPPED — {row.get('note', 'unknown reason')}")
            print(f"  ► Cointegration verdict: UNDETERMINED")
            continue

        n_spells = int(row.get('n_spells_used', 0))
        print(f"  Spell ADF + Fisher (Stage 1B)")
        print(f"    Spells ≥ 40 obs   : {n_spells}  (each tested independently to avoid cross-spell ADF contamination)")
        print(f"    Fisher χ²         : {row['fisher_chi2']:.4f}  (sum statistic: −2·Σln(pᵢ), df={2*n_spells})")
        print(f"    Combined p-value  : {row['fisher_pval']:.4f}  (H₀: unit root in ALL spells)")

        if row['cointegrated']:
            print(f"  ► Cointegration verdict: COINTEGRATED  ✓  (p={row['fisher_pval']:.4f} < 0.05)")
            print(f"    Interpretation: Spread {pair_key} recurrently reverts to equilibrium within Regime {r} spells.")
        else:
            print(f"  ► Cointegration verdict: NOT COINTEGRATED  ✗  (p={row['fisher_pval']:.4f} ≥ 0.05)")
            print(f"    Interpretation: Cannot reject that the spread is a unit-root process in Regime {r}.")
            print(f"    Spread trades in this regime lack a statistical mean-reversion anchor.")


══════════════════════════════════════════════════════════════════════
  PAIR: USD-EUR 5Y
══════════════════════════════════════════════════════════════════════

  ── Regime 0 ──────────────────────────────────────────────
  OLS (Stage 1A)
    Observations used : 2,241
    Hedge ratio β     : 0.6591  (short 0.6591 units of y_B per unit of y_A)
    Intercept α       : 1.5932  (regime equilibrium spread level)
    R²                : 0.6224  (proportion of y_A variation explained by y_B)
  Spell ADF + Fisher (Stage 1B)
    Spells ≥ 40 obs   : 16  (each tested independently to avoid cross-spell ADF contamination)
    Fisher χ²         : 24.9882  (sum statistic: −2·Σln(pᵢ), df=32)
    Combined p-value  : 0.8065  (H₀: unit root in ALL spells)
  ► Cointegration verdict: NOT COINTEGRATED  ✗  (p=0.8065 ≥ 0.05)
    Interpretation: Cannot reject that the spread is a unit-root process in Regime 0.
    Spread trades in this regime lack a statistical mean-reversion anchor.

  ── Regime 1 ─────────

## Stage 2: Mean Reversion Speed and Tradability

For regimes where cointegration was confirmed, we now estimate **how fast** the spread reverts. Speed matters because a trade can only be profitable if the regime persists long enough to allow the spread to complete its journey back to equilibrium.

### The AR(1) Mean Reversion Model

We fit the discrete-time Ornstein-Uhlenbeck equation:

$$\Delta s_t = c + \alpha \cdot s_{t-1} + u_t$$

where $s_t = y_{A,t} - \hat{\alpha}_r - \hat{\beta}_r \cdot y_{B,t}$ is the regime-specific spread.

- **$\hat{\alpha}$ (AR(1) coefficient):** Must be negative for mean reversion. The more negative, the faster the reversion.
- **Half-life:** The expected time for the spread to close half the distance to its mean, given by:

$$\text{HL} = \frac{-\ln(2)}{\hat{\alpha}} \text{ (in trading days)}$$

For example, $\hat{\alpha} = -0.05$ → HL = 13.9 days.

**Implementation details:**
- Each contiguous spell is **de-meaned independently** before pooling. This prevents heterogeneous equilibrium levels across different historical episodes from biasing the estimate of $\hat{\alpha}$ upward (which would produce a spuriously fast mean reversion).
- HAC standard errors (`maxlags=5`) correct for residual autocorrelation in the inference on $\hat{\alpha}$.
- The pooled regression has no re-estimated cointegrating vector — it uses $\hat{\alpha}_r$, $\hat{\beta}_r$ from Stage 1A throughout.

### Tradability Decision

A regime is classified as **tradable** if all three conditions hold simultaneously:

| Condition | Threshold | Rationale |
|---|---|---|
| $\hat{\alpha} < 0$ | — | Mean reversion direction confirmed |
| p-value on $\hat{\alpha}$ | < 0.05 | Statistically significant reversion |
| P(regime duration ≥ HL) | > 60% | Majority of regime spells are long enough to realise the trade |

The persistence condition is the key practical filter. A regime may be statistically mean-reverting but composed of very short spells — the trade would be terminated by a regime switch before the spread closes. Requiring that >60% of historical spells exceed the half-life provides a structural buffer.

In [44]:
all_pair_results = []

for pair_key, data in yield_data.items():
    coint_df = all_coint_results[pair_key]

    print(f"\n{'═'*70}")
    print(f"  PAIR: {pair_key}  —  Mean Reversion & Tradability")
    print(f"{'═'*70}")

    for _, row in coint_df.iterrows():
        r = int(row['regime_state'])
        print(f"\n  ── Regime {r} ──────────────────────────────────────────────")

        if not row.get('cointegrated'):
            verdict = row.get('note') or 'Not cointegrated in Stage 1'
            print(f"  SKIPPED — {verdict}")
            print(f"  No mean reversion analysis performed (spread lacks a mean to revert to).")
            continue

        mr = estimate_pooled_mean_reversion(
            y_A=data['y_A'],
            y_B=data['y_B'],
            alpha_r=row['alpha'],
            beta_r=row['hedge_ratio_beta'],
            regime_series=df['regime_most_likely'],
            target_state=r
        )

        if not mr['valid']:
            print(f"  Mean reversion estimation failed: {mr['reason']}")
            continue

        hl        = mr['half_life']
        durs      = np.array(durations_map[r])
        median_dur = np.median(durs)
        prob_persist = np.mean(durs >= hl) if np.isfinite(hl) else 0.0

        # Condition flags
        cond_direction   = mr['alpha'] < 0
        cond_significant = mr['p_value'] < 0.05
        cond_persistent  = prob_persist > 0.4
        is_tradable      = cond_direction and cond_significant and cond_persistent

        print(f"  AR(1) Mean Reversion")
        print(f"    α (reversion coeff)  : {mr['alpha']:.5f}  {'✓ negative (mean-reverting)' if cond_direction else '✗ non-negative (diverging or random walk)'}")
        print(f"    p-value on α         : {mr['p_value']:.4f}  {'✓ significant at 5%' if cond_significant else '✗ not significant'}")
        print(f"    Half-life            : {hl:.1f} trading days  ({'≈ ' + str(round(hl/5, 1)) + ' weeks' if np.isfinite(hl) else 'infinite'})")
        print(f"    Pooled observations  : {mr['n_obs']:,}")

        print(f"\n  Persistence Check")
        print(f"    Median regime duration         : {median_dur:.0f} days")
        print(f"    P(spell duration ≥ half-life)  : {prob_persist:.1%}  {'✓ > 40%' if cond_persistent else '✗ ≤ 40% — most spells too short to trade'}")
        print(f"    Duration distribution          : min={durs.min():.0f}d, median={median_dur:.0f}d, max={durs.max():.0f}d")

        print(f"\n  ► Tradability verdict: {'TRADABLE ✓' if is_tradable else 'NOT TRADABLE ✗'}")
        if not is_tradable:
            reasons = []
            if not cond_direction:   reasons.append("α is non-negative")
            if not cond_significant: reasons.append(f"α is not significant (p={mr['p_value']:.3f})")
            if not cond_persistent:  reasons.append(f"only {prob_persist:.0%} of spells exceed the half-life of {hl:.0f}d")
            print(f"    Reason(s): {'; '.join(reasons)}")

        all_pair_results.append({
            'pair_id': pair_key,
            'y_col': data['col_A'],
            'x_col': data['col_B'],
            'tenor': data['tenor'],            
            'ccy_leg_1': data['ccy_A'],          
            'ccy_leg_2': data['ccy_B'],        
            'tradable_regimes': f"[{r}]",      
            'cointegration_source': 'HMM',   
            'status': 'approved' if is_tradable else 'rejected', 
            'regime': r,
            'fisher_pval': row['fisher_pval'],
            'beta_r': row['hedge_ratio_beta'],
            'alpha_r': row['alpha'],
            'ar1_alpha': mr['alpha'], 
            'ar1_pval': mr['p_value'],
            'half_life': hl, 
            'median_duration': median_dur,
            'prob_persistence': prob_persist, 
            'is_tradable': is_tradable
        })


══════════════════════════════════════════════════════════════════════
  PAIR: USD-EUR 5Y  —  Mean Reversion & Tradability
══════════════════════════════════════════════════════════════════════

  ── Regime 0 ──────────────────────────────────────────────
  SKIPPED — Not cointegrated in Stage 1
  No mean reversion analysis performed (spread lacks a mean to revert to).

  ── Regime 1 ──────────────────────────────────────────────
  SKIPPED — Not cointegrated in Stage 1
  No mean reversion analysis performed (spread lacks a mean to revert to).

  ── Regime 2 ──────────────────────────────────────────────
  SKIPPED — Not cointegrated in Stage 1
  No mean reversion analysis performed (spread lacks a mean to revert to).

══════════════════════════════════════════════════════════════════════
  PAIR: USD-EUR 10Y  —  Mean Reversion & Tradability
══════════════════════════════════════════════════════════════════════

  ── Regime 0 ──────────────────────────────────────────────
  SKIPPED — Not 

## Summary Table

The table below consolidates all regime-pair combinations that passed the cointegration screen into a single tradability scorecard.

Column guide:
- **fisher_pval**: Stage 1B combined p-value (lower = stronger cointegration evidence)
- **beta_r**: Regime-specific hedge ratio
- **ar1_alpha**: Mean-reversion coefficient (more negative = faster reversion)
- **half_life**: Expected days to close half the spread gap
- **prob_persistence**: Fraction of historical regime spells that last at least as long as the half-life
- **is_tradable**: `True` only if all three tradability conditions are met

In [45]:
"""
df_results = pd.DataFrame(all_pair_results)

if len(df_results) > 0:
    display_cols = ['pair', 'regime', 'fisher_pval', 'beta_r', 'alpha_r',
                    'ar1_alpha', 'ar1_pval', 'half_life', 'median_duration',
                    'prob_persistence', 'is_tradable']
    print(df_results[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
else:
    print("No cointegrated regimes found across any pair.")

df_results.to_csv(results_dir / 'regime_tradability_results.csv', index=False)
print(f"\nSaved to {results_dir / 'regime_tradability_results.csv'}")
"""
df_results = pd.DataFrame(all_pair_results)

if len(df_results) > 0:
    # Notice we use 'pair_id' here now to match the dataframe
    display_cols = ['pair_id', 'regime', 'fisher_pval', 'beta_r', 'alpha_r',
                    'ar1_alpha', 'ar1_pval', 'half_life', 'median_duration',
                    'prob_persistence', 'is_tradable']
    
    print("\n" + "═"*100)
    print(" FINAL REGIME-CONDITIONAL TRADABILITY SUMMARY")
    print("" + "═"*100)
    print(df_results[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    
    df_results.to_csv(results_dir / 'regime_tradability_results.csv', index=False)
    print(f"\nSaved full diagnostic report to: {results_dir / 'regime_tradability_results.csv'}")

    approved_df = df_results[df_results['is_tradable'] == True].copy()
    
    if not approved_df.empty:
        # Final rename to match the Contract Loader's exact variable names
        approved_df = approved_df.rename(columns={
            'beta_r': 'hedge_ratio',
            'alpha_r': 'intercept'
        })

        approved_df.to_csv(results_dir / 'approved_pairs.csv', index=False)
        print(f"PIPELINE SUCCESS: {len(approved_df)} tradable pairs sent to 'approved_pairs.csv'")
    else:
        print("PIPELINE ALERT: No pairs passed the Tradability filter.")

else:
    print("No cointegrated regimes found across any pair.")


════════════════════════════════════════════════════════════════════════════════════════════════════
 FINAL REGIME-CONDITIONAL TRADABILITY SUMMARY
════════════════════════════════════════════════════════════════════════════════════════════════════
   pair_id  regime  fisher_pval  beta_r  alpha_r  ar1_alpha  ar1_pval  half_life  median_duration  prob_persistence  is_tradable
JPY-AUD 5Y       0       0.0457  0.2505  -0.4653    -0.0833    0.0000     8.3247           5.0000            0.4052         True

Saved full diagnostic report to: C:\Users\kiefe\OneDrive\Documents\GitHub\Cross-Country-Rates-Relative-Value-NUSInvest2526\results\regime_tradability_results.csv
PIPELINE SUCCESS: 1 tradable pairs sent to 'approved_pairs.csv'


## Questions
1. Why is the min_spell_obs = 40?
2. What is the difference between schwert lag method and aic lag method